# Агрегаты, объединения, признаки текста и времени

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Строить агрегаты и признаки из доступных записей, текста и времени.

## Признак — вычисление по сведениям

Вместо одного события можно описать историю объекта: число прошлых посещений, среднее время предыдущих попыток, дни с последнего входа. Такие показатели называются агрегатами. Для них важны единица группировки и момент отсечения истории. Среднее по пользователю не должно включать его будущие события.

Для текста простыми признаками могут быть длина, число слов и наличие заранее выбранного выражения. Для времени — день недели или час. Такие признаки отражают лишь часть смысла; число символов не понимает содержание сообщения, а номер месяца не задаёт непрерывную прямую шкалу сезонности.

## Разобранный пример: прогноз перед третьим днём

У ученика были результаты 2 в день 1, 4 в день 2 и 100 в день 4. Перед днём 3 доступно среднее (2+4)/2=3. Среднее всех трёх равно 106/3 и содержит будущую запись. Признак «средний результат» без указания временного окна неоднозначен.

При объединении справочника нужна проверка ключа. Если одному пользователю соответствуют несколько строк справочника, обычное объединение может размножить объекты. В pandas validate='many_to_one' требует уникальность правого ключа, а indicator=True позволяет проверить отсутствие соответствия.

## Проверяем готовый столбец

Для каждого признака запишите источник, правило, доступное время и обработку отсутствующей истории. Если истории нет, нулевое число посещений осмысленно, но нулевое среднее результата может быть ложным измерением. Оставьте пропуск и отдельный индикатор либо задайте другое обоснованное правило.

В ноутбуке фильтр по дню выполняется до groupby. Затем среднее добавляется к списку целевых объектов левым объединением: новый пользователь остаётся строкой с неизвестной историей.

В вызове `groupby(..., as_index=False)` параметр `as_index=False` оставляет ключ группы обычным столбцом результата. Так таблицу средних удобно присоединить через `merge` по этому ключу. Если ключ оказался индексом, способ объединения пришлось бы изменить.

## Разобранный пример: проверяем вычисление

Все числа в примере придуманы для этого урока. Выполните код, затем измените вход и объясните новый результат.

In [ ]:
import pandas as pd
history = pd.DataFrame({"user": ["a", "a", "a"], "day": [1, 2, 4], "value": [2., 4., 100.]})
available = history.loc[history["day"] < 3]
means = available.groupby("user", as_index=False)["value"].mean()
objects = pd.DataFrame({"user": ["a", "b"]})
result = objects.merge(means, on="user", how="left", validate="many_to_one")
assert result.loc[0, "value"] == 3
assert pd.isna(result.loc[1, "value"])
assert len(result) == 2
print(result)


## Типичные ошибки

Сначала временной фильтр, затем агрегат. Не допускайте размножения строк при join и не превращайте неизвестное среднее в измеренный ноль.

## Задачи

### 1. Базовая

Что называется агрегатом?

<details>
<summary>Подсказка</summary>

Несколько записей дают один показатель.

</details>



### 2. Базовая

Почему у признака нужен момент отсечения?

<details>
<summary>Подсказка</summary>

Записи могут появляться позже прогноза.

</details>



### 3. Прикладная

Результаты до дня 5: 2 и 6. Найдите среднее истории.

<details>
<summary>Подсказка</summary>

Не включайте более поздние записи.

</details>



### 4. Прикладная

У нового пользователя нет истории. Можно ли сохранить строку левым объединением?

<details>
<summary>Подсказка</summary>

Левые объекты не теряются.

</details>



### 5. Олимпиадная

Почему validation пользователя нельзя включать в его обучающий агрегат по ответу?

<details>
<summary>Подсказка</summary>

Правильный ответ проверки станет частью признака.

</details>



### 6. Олимпиадная

Почему среднее часа 23 и 1 не всегда полезно?

<details>
<summary>Подсказка</summary>

Время суток циклично.

</details>



## Итоги и самопроверка

Для среднего истории укажите ключ, окно и дату доступности. Проверьте новый объект без событий и объект с поздней записью.

Для перехода дальше достаточно базовых и прикладных задач. Олимпиадные задачи можно отложить.


## Решения задач

### Задача 1

Сводное вычисление по группе, например сумма или среднее прошлых событий.

### Задача 2

Чтобы использовать только сведения, реально доступные в момент ответа.

### Задача 3

(2+6)/2=4.

### Задача 4

Да; агрегат будет пропущен, что нужно явно обработать.

### Задача 5

Это утечка: признак несёт сведения, которые модель должна была предсказать.

### Задача 6

Арифметическое среднее 12 пропускает близость этих часов через полночь. Представление должно учитывать смысл шкалы.